In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
!pip install mlflow boto3 awscli optuna xgboost imbalanced-learn
# !aws configure

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0

In [ ]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/")

In [ ]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

2026/01/25 09:24:47 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 - ML Algos with HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-bucket-1712/5', creation_time=1769333087920, experiment_id='5', last_update_time=1769333087920, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}>

In [ ]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [ ]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

ngram_range = (1, 3)  # Trigram setting
max_features = 10000  # Set max_features to 1000 for TF-IDF

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

# Step 2: Vectorization using TF-IDF, fit on training data only
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
X_test_vec = vectorizer.transform(X_test)  # Transform test data

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))


# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)

# Run the experiment for XGBoost
run_optuna_experiment()

[I 2026-01-25 13:10:26,074] A new study created in memory with name: no-name-ef26e5b6-405e-4e10-931a-87004a420924
[I 2026-01-25 13:11:29,125] Trial 0 finished with value: 0.6737638571177195 and parameters: {'n_estimators': 270, 'learning_rate': 0.00035310741268384783, 'max_depth': 5}. Best is trial 0 with value: 0.6737638571177195.
[I 2026-01-25 13:12:21,471] Trial 1 finished with value: 0.7115959880344889 and parameters: {'n_estimators': 84, 'learning_rate': 0.0011603825943049834, 'max_depth': 10}. Best is trial 1 with value: 0.7115959880344889.
[I 2026-01-25 13:12:31,897] Trial 2 finished with value: 0.7332394861868732 and parameters: {'n_estimators': 61, 'learning_rate': 0.022121934949421936, 'max_depth': 4}. Best is trial 2 with value: 0.7332394861868732.
[I 2026-01-25 13:12:42,973] Trial 3 finished with value: 0.6815062467006863 and parameters: {'n_estimators': 108, 'learning_rate': 0.010228185448133928, 'max_depth': 3}. Best is trial 2 with value: 0.7332394861868732.
[I 2026-01-2

🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5/runs/cc6f14a252dd41a4a923d4e38450365c
🧪 View experiment at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5


In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier

# General objective runner for Optuna
def run_optuna_generic(model_name, objective_func, best_model_builder, n_trials=30):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_func, n_trials=n_trials)

    best_params = study.best_params
    best_model = best_model_builder(best_params)

    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # Log tuned hyperparameters
        for k, v in best_params.items():
            mlflow.log_param(k, v)

        # Train timing
        start_train = time.time()
        best_model.fit(X_train_vec, y_train)
        train_time = time.time() - start_train

        # Inference timing
        start_pred = time.time()
        y_pred = best_model.predict(X_test_vec)
        infer_time = time.time() - start_pred

        accuracy = accuracy_score(y_test, y_pred)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("train_time_sec", train_time)
        mlflow.log_metric("inference_time_sec", infer_time)

        mlflow.sklearn.log_model(best_model, name=f"{model_name}_model")

        print(f"{model_name} | Accuracy: {accuracy:.4f} | Train Time: {train_time:.2f}s | Infer Time: {infer_time:.4f}s")

    return best_model


# ----------------- DECISION TREE -----------------

def objective_dt(trial):
    max_depth = trial.suggest_int("max_depth", 3, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))

def dt_builder(params):
    return DecisionTreeClassifier(**params, random_state=42)

run_optuna_generic("Decision_Tree", objective_dt, dt_builder)


# ----------------- KNN -----------------

def objective_knn(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 3, 15)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])

    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        n_jobs=-1
    )

    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))

def knn_builder(params):
    return KNeighborsClassifier(**params, n_jobs=-1)

run_optuna_generic("KNN", objective_knn, knn_builder)


# ----------------- NAIVE BAYES -----------------

def objective_nb(trial):
    alpha = trial.suggest_float("alpha", 0.01, 1.0, log=True)

    model = MultinomialNB(alpha=alpha)

    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))

def nb_builder(params):
    return MultinomialNB(**params)

run_optuna_generic("Naive_Bayes", objective_nb, nb_builder)


[I 2026-01-25 12:57:27,420] A new study created in memory with name: no-name-0fdb410d-be97-4638-9732-dddc07c3d8eb
[I 2026-01-25 12:57:32,211] Trial 0 finished with value: 0.6328924042001909 and parameters: {'max_depth': 26, 'min_samples_split': 15}. Best is trial 0 with value: 0.6328924042001909.
[I 2026-01-25 12:57:36,393] Trial 1 finished with value: 0.6072548752216009 and parameters: {'max_depth': 20, 'min_samples_split': 12}. Best is trial 0 with value: 0.6328924042001909.
[I 2026-01-25 12:57:41,551] Trial 2 finished with value: 0.6278467203054684 and parameters: {'max_depth': 24, 'min_samples_split': 5}. Best is trial 0 with value: 0.6328924042001909.
[I 2026-01-25 12:57:44,432] Trial 3 finished with value: 0.5760261830083185 and parameters: {'max_depth': 16, 'min_samples_split': 5}. Best is trial 0 with value: 0.6328924042001909.
[I 2026-01-25 12:57:45,651] Trial 4 finished with value: 0.5049774989772262 and parameters: {'max_depth': 8, 'min_samples_split': 8}. Best is trial 0 wi

Decision_Tree | Accuracy: 0.6460 | Train Time: 6.24s | Infer Time: 0.0024s
🏃 View run Decision_Tree_SMOTE_TFIDF_Trigrams at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5/runs/eb78293c522b449089207b40553b14d7
🧪 View experiment at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5


[I 2026-01-25 13:00:02,279] A new study created in memory with name: no-name-76878f11-d671-4e30-891c-fb60c140b08f
[I 2026-01-25 13:00:23,997] Trial 0 finished with value: 0.4182462839219965 and parameters: {'n_neighbors': 8, 'weights': 'distance'}. Best is trial 0 with value: 0.4182462839219965.
[I 2026-01-25 13:00:41,294] Trial 1 finished with value: 0.43542888313105144 and parameters: {'n_neighbors': 3, 'weights': 'uniform'}. Best is trial 1 with value: 0.43542888313105144.
[I 2026-01-25 13:00:59,659] Trial 2 finished with value: 0.3934269739533615 and parameters: {'n_neighbors': 13, 'weights': 'distance'}. Best is trial 1 with value: 0.43542888313105144.
[I 2026-01-25 13:01:17,843] Trial 3 finished with value: 0.408291285967544 and parameters: {'n_neighbors': 10, 'weights': 'distance'}. Best is trial 1 with value: 0.43542888313105144.
[I 2026-01-25 13:01:35,428] Trial 4 finished with value: 0.39233601527342155 and parameters: {'n_neighbors': 14, 'weights': 'distance'}. Best is trial

KNN | Accuracy: 0.4384 | Train Time: 0.01s | Infer Time: 18.5571s
🏃 View run KNN_SMOTE_TFIDF_Trigrams at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5/runs/4d49b2894d9d4fb4b5c2453783f4d99a
🧪 View experiment at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5


[I 2026-01-25 13:09:45,487] A new study created in memory with name: no-name-ab80f79b-f62f-4b79-a652-67092068b94d
[I 2026-01-25 13:09:45,502] Trial 0 finished with value: 0.7098049911359607 and parameters: {'alpha': 0.05806256321348577}. Best is trial 0 with value: 0.7098049911359607.
[I 2026-01-25 13:09:45,515] Trial 1 finished with value: 0.7081685531160508 and parameters: {'alpha': 0.09672627859386132}. Best is trial 0 with value: 0.7098049911359607.
[I 2026-01-25 13:09:45,528] Trial 2 finished with value: 0.7093958816309832 and parameters: {'alpha': 0.015448752468931884}. Best is trial 0 with value: 0.7098049911359607.
[I 2026-01-25 13:09:45,547] Trial 3 finished with value: 0.7104868403109232 and parameters: {'alpha': 0.04428964200264536}. Best is trial 3 with value: 0.7104868403109232.
[I 2026-01-25 13:09:45,563] Trial 4 finished with value: 0.7096686213009682 and parameters: {'alpha': 0.05686315129396008}. Best is trial 3 with value: 0.7104868403109232.
[I 2026-01-25 13:09:45,57

Naive_Bayes | Accuracy: 0.7108 | Train Time: 0.01s | Infer Time: 0.0013s
🏃 View run Naive_Bayes_SMOTE_TFIDF_Trigrams at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5/runs/7b83cbdd51814ed29af04afaf97acefd
🧪 View experiment at: http://ec2-44-222-234-130.compute-1.amazonaws.com:5000/#/experiments/5


MultinomialNB(alpha=0.03794123004612616)